# RG3 gas: keep baseline and guard only additional expert detections
Original baseline frozen; new branch requires expert high score AND baseline above prespecified train-normal percentile (50/75/90).
Nine prespecified candidates. Thresholds from inner held-out normals. Train-only selector deploys addition only if inner TP gain>=1, added FP<=2, F1 improves; otherwise baseline fallback.
Reuse raw scores from42, no new model training. All original positives guaranteed preserved. Candidate best after observing outer results is exploratory.
Cannot interpret baseline/expert evidence as statistically independent; they share features and data.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
outer=pd.read_csv(tables/'rg3_rank_fusion_predictions.csv',parse_dates=['TimeStamp'])
outer=outer.loc[outer.model.eq('baseline_rank')].copy()
inside=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
original=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
original=original.loc[original.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
records=[];choices=[]
gates=[.50,.75,.90]
rates=[.0025,.005,.01]
def stats(yy,p):
    tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
    return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
for (scheme,b),q in outer.groupby(['scheme','block'],sort=False):
    cal=inside.loc[inside.scheme.eq(scheme)&inside.heldout_block.eq(b)].dropna(subset=['raw_baseline_score','raw_expert_score']).reset_index(drop=True)
    n=cal.actual.eq(0)
    cut_b=float(np.quantile(cal.loc[n,'raw_baseline_score'],.99,method='higher'))
    base_inner=cal.raw_baseline_score.gt(cut_b)
    base_outer=q.raw_baseline_score.gt(cut_b)
    tp0,fp0,fn0,tn0,f0=stats(cal.actual,base_inner)
    candidates=[]
    for gate in gates:
        cut_gate=float(np.quantile(cal.loc[n,'raw_baseline_score'],gate,method='higher'))
        for rate in rates:
            cut_e=float(np.quantile(cal.loc[n,'raw_expert_score'],1-rate,method='higher'))
            pr=base_inner|(cal.raw_expert_score.gt(cut_e)&cal.raw_baseline_score.gt(cut_gate))
            out=base_outer|(q.raw_expert_score.gt(cut_e)&q.raw_baseline_score.gt(cut_gate))
            tp,fp,fn,tn,f=stats(cal.actual,pr)
            name=f'gate{gate}_expert_fpr{rate}'
            c=dict(model=name,baseline_threshold=cut_b,expert_threshold=cut_e,gate_threshold=cut_gate,inner_F1=f,inner_TP=tp,inner_FP=fp,inner_added_TP=tp-tp0,inner_added_FP=fp-fp0)
            candidates.append((c,out))
            for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model=name,TimeStamp=r.TimeStamp,block=b,actual=int(r.actual),prediction=int(out.iloc[j]),baseline_prediction=int(base_outer.iloc[j])))
    # Only deploy extra branch if inner sensitivity gain is seen at <=2 added false normals.
    eligible=[(c,p) for c,p in candidates if c['inner_added_TP']>=1 and c['inner_added_FP']<=2 and c['inner_F1']>f0]
    if eligible:
        selected,out=max(eligible,key=lambda cp:(cp[0]['inner_F1'],-cp[0]['inner_added_FP'],cp[0]['gate_threshold']))
    else:
        selected=dict(model='baseline_only',baseline_threshold=cut_b,expert_threshold=np.nan,gate_threshold=np.nan,inner_F1=f0,inner_TP=tp0,inner_FP=fp0,inner_added_TP=0,inner_added_FP=0)
        out=base_outer
    choices.append(dict(scheme=scheme,block=b,**selected))
    for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model='inner_guarded_selection',TimeStamp=r.TimeStamp,block=b,actual=int(r.actual),prediction=int(out.iloc[j]),baseline_prediction=int(base_outer.iloc[j])))
pp=pd.DataFrame(records)
check=pp.loc[pp.model.eq('inner_guarded_selection')].merge(original,on=['scheme','TimeStamp'],validate='one_to_one')
assert len(check)==1023 and (check.baseline_prediction.astype(bool)==check.predicted_train_fpr1).all()
assert (check.prediction>=check.baseline_prediction).all()
rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tp,fp,fn,tn,f=stats(g.actual,g.prediction)
    rows.append(dict(scheme=scheme,model=name,TP=tp,FP=fp,FN=fn,TN=tn,F1=f))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_guarded_expert_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_guarded_expert_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_guarded_expert_inner_choices.csv',index=False,encoding='utf-8-sig')
for scheme in metrics.scheme.unique():
    m=metrics.loc[metrics.scheme.eq(scheme)]
    print(scheme,m.sort_values(['F1','FP'],ascending=[False,True]).head(10).to_string(index=False))


## Executed results
Fixed gate0.5/expert_fpr0.005 whole TP5 FP11 F1=0.303; forward TP3 FP6 F1=0.273. Internal guarded selection whole TP4 FP7 F1=0.286, forward TP2 FP5 F1=0.200. All baseline detections retained. No promotion.
See reports/rg3_iterative_improvement_log.md for protocol, limitations and source tables.